# SMI Annotation Experiment — Automated Labeling via Swarabyanjan Multi-criteria Index

**Authors:** Swagotam Malakar, Anamika Das, Dr. Ohidujjaman

## Overview

This notebook demonstrates SMI (Swarabyanjan Multi-criteria Index) as an **automated annotation framework** for Bengali yellow journalism detection. SMI uses 8 linguistic criteria scoring functions to automatically label news articles without human annotators.

### Reproducibility

All experiments use seed=42 and pinned dependencies.

### Evaluation protocol

SMI is evaluated at three tiers, in increasing strength of the generalization claim:

1. **5-fold CV on 766 gold** (F1 ≈ 0.809 ± 0.056) — variance estimate from stratified 5-fold cross-validation on the gold standard. Reported as the secondary metric because each fold's training set overlaps with the union of the other folds' training sets.
2. **70/30 held-out on 766 gold** (primary in-sample held-out) — a single 70/30 stratified split (536 train / 230 test). The decision threshold is chosen on the TRAIN split only (no test leakage). The held-out F1 / Kappa / AUC are the PRIMARY reported generalization metrics.
3. **4,234 non-gold articles vs `MY_LABEL`** (true out-of-sample, lower bound) — the 5,000-article corpus contains 766 articles that overlap with the gold standard (used for SMI training) and 4,234 articles SMI has never seen. The corpus CSV provides a human annotation (`MY_LABEL`) for all 5,000 articles, which lets us evaluate SMI on a true held-out set. `MY_LABEL` is the *pre-revision* annotation; it agrees with `best_label` only **59.9%** of the time on the 766 overlap, so SMI's F1 against `MY_LABEL` on the 4,234 non-gold articles is a **lower bound** on SMI's true generalization performance. See Section 6.5 for the ceiling-adjusted estimate.

### Corpus file note

The 5,000-article corpus is `v18_HUMAN_GOLD_FINAL_v20_GOLD_5000.csv` from the Kaggle dataset `v18-human-gold-final` (https://www.kaggle.com/datasets/swagotammalakar/v18-human-gold-final). This is the same dataset that hosts the 766-article gold standard `Swarabyanjan_BEST_BALANCED_1to1.csv`. The corpus CSV uses different column names than the gold standard — the setup cell renames them as follows:

| Corpus CSV column | Renamed to | Notes |
|---|---|---|
| `v18_id` | `article_id` | Article ID (e.g. `v18_1501`) |
| `headline` | `headline` | Bengali headline (no rename) |
| `body_preview` | `body_text` | Bengali body text — **truncated at 1200 chars** when `body_full_len > 1200` (affects ~9.1% of articles; see Section 5 printout for exact count) |
| `body_full_len` | `article_length` | True body length |
| `source` | `news_source` | Article source |
| `MY_LABEL` | `human_label` | Pre-revision human annotation (0/1) — 2702 zeros + 2298 ones |
| `MY_CONFIDENCE` | `human_confidence` | Annotation confidence (H/M/L) |
| `MY_NOTE` | `human_note` | Annotation note |

Columns `weak_label`, `model_prediction`, `cosine_similarity`, `cos_band`, `error_cell`, `llm_consensus`, `llm_vote_type`, `llm_voters_summary` are ignored (they are noisy intermediate signals from the corpus-construction pipeline).

### Experiment Design

1. **Training set:** 766 human-verified articles (383 yellow + 383 non-yellow) — used to learn SMI weights
2. **Held-out evaluation (PRIMARY):** 70/30 stratified split of the gold standard (536 train / 230 test). The held-out F1 / Kappa / AUC are the primary generalization metrics reported.
3. **5-fold cross-validation (SECONDARY):** Reported alongside the held-out number for a variance estimate on the same 766 articles.
4. **True out-of-sample (STRONGEST):** SMI evaluated against `MY_LABEL` on the 4,234 corpus articles that do NOT overlap with the gold standard. See Section 6.5.
5. **Annotation target:** All 5,000 articles in the corpus — SMI generates labels automatically using a model trained on ALL 766 gold articles.
6. **Scalability:** Measure annotation time (SMI: seconds vs. human: hours)

### SMI Mathematical Framework

For an article $a = (h, b)$ with headline $h$ and body $b$:

$$\text{SMI}(a) = \sigma\left(\sum_{i=1}^{8} w_i \cdot C_i(a) + w_0\right) \in [0, 1]$$

where:
- $C_i(a)$: Score of criterion $i$ for article $a$ (8 linguistic criteria)
- $w_i$: Learned weight for criterion $i$
- $w_0$: Bias term
- $\sigma$: Sigmoid function

**Decision rule:** $\hat{y}(a) = 1$ if $\text{SMI}(a) \geq \tau^*$, else $0$

**Weight learning:** $\min_{\mathbf{w}} \frac{1}{n}\sum_{j=1}^{n} \mathcal{L}_{\text{log}}(y_j, \text{SMI}(a_j)) + \lambda\|\mathbf{w}\|_2^2$

### Data version note

This notebook supports **both** the cleaned CSVs ( output) and the legacy CSVs:

| CSV | Cleaned filename (recommended) | Legacy filename (backward compatible) |
|---|---|---|
| Gold standard (766 articles) | `Swarabyanjan_Gold_Balanced_766.csv` | `Swarabyanjan_BEST_BALANCED_1to1.csv` |
| Corpus main (5,000 articles) | `Swarabyanjan_Corpus_5000.csv` | `v18_HUMAN_GOLD_FINAL_v20_GOLD_5000.csv` |
| Corpus metadata (optional) | `Swarabyanjan_Corpus_Metadata_5000.csv` | *(no legacy equivalent)* |

**The cleaned CSVs are recommended for publication.** They differ from the legacy CSVs in three ways:

1. **NFC normalization** — all Bengali text is normalized to Unicode NFC form (fixes 1,106 cells in gold + 7,422 cells in corpus that were not NFC-normalized).
2. **ZWJ/ZWNJ stripping** — zero-width joiners (U+200D) and zero-width non-joiners (U+200C) are stripped from body text (fixes inconsistent tokenization around Bengali conjuncts like "র্যাব" / RAB — Rapid Action Battalion). Affects 238 gold rows + 1,571 corpus rows.
3. **Column renames for professionalism** — see the rename table below.

**Column rename table (legacy → cleaned):**

| Legacy column | Cleaned column | Notes |
|---|---|---|
| `news_source` (gold) | `corpus_batch` | Contains corpus-construction batch tags (not news outlet names) |
| `v18_id` (corpus) | `article_id` | Unified with gold schema |
| `body_preview` (corpus) | `body_text` | Unified with gold schema |
| `body_full_len` (corpus) | `article_length` | Unified with gold schema |
| `source` (corpus) | `corpus_batch` | Unified with gold schema |
| `MY_LABEL` (corpus) | `human_label` | Drops unprofessional "MY_" prefix |
| `MY_CONFIDENCE` (corpus) | `human_confidence` | Drops unprofessional "MY_" prefix |
| `MY_NOTE` (corpus) | `annotation_provenance` | More descriptive name |

The notebook auto-detects which schema is in use via the `detect_gold_columns()` function (cell 5) and the schema-detection block in cell 12. The `using_cleaned` flag (cell 1) is recorded in the output `smi_annotation_results.json` under the `data_version` key for full provenance.

**Stub articles (body_text = "not_available"):** Both CSVs contain a small number of legitimate "stub article" cases (2 in gold, 4 in corpus) where the article body could not be scraped (article_length = 0). These are labeled yellow via C7 (headline-body mismatch) and C2 (clickbait) — they are NOT data errors. The notebook preserves them in the dataset (the original "not_available" string is kept in `body_text_original`) but replaces them with an empty string for SMI density-based scoring (C3/C5/C6), which gracefully return 0.0 for empty bodies. An offline audit was performed (report not shipped) for the full list of stub article IDs.


In [1]:
# === SETUP ===
import os, sys, time, json, warnings, glob
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, cohen_kappa_score, matthews_corrcoef,
                             roc_auc_score, confusion_matrix, classification_report)
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

# === Dataset path discovery ===
# Task 8 (2026-07-19): Cleaned CSVs are now the primary input. They apply NFC
# normalization, strip ZWJ/ZWNJ, and use professional column names.
# Task 9 (2026-07-19): NB8 now supports BOTH the cleaned CSVs (primary) and the
# legacy CSVs (backward compatible). The find_file() function tries the primary
# filename first, then falls back to the legacy filename.

# Primary (cleaned) filenames — Task 8's output
GOLD_FILENAME = 'Swarabyanjan_Gold_Balanced_766.csv'
CORPUS_FILENAME = 'Swarabyanjan_Corpus_5000.csv'
CORPUS_METADATA_FILENAME = 'Swarabyanjan_Corpus_Metadata_5000.csv'  # optional

# Legacy filenames (for backward compatibility with old Kaggle datasets)
GOLD_FILENAME_LEGACY = 'Swarabyanjan_BEST_BALANCED_1to1.csv'
CORPUS_FILENAME_LEGACY = 'v18_HUMAN_GOLD_FINAL_v20_GOLD_5000.csv'

def find_file(filename, legacy_filename=None):
    """Find a file in Kaggle input paths or local paths.
    Tries the primary filename first, then the legacy filename if provided.
    Returns the path to the first file found, or the primary filename if
    nothing is found (will fail later with a helpful error from the loader).
    """
    filenames_to_try = [filename]
    if legacy_filename:
        filenames_to_try.append(legacy_filename)

    for fname in filenames_to_try:
        candidates = [
            f'/kaggle/input/v18-human-gold-final/{fname}',  # Primary Kaggle dataset (has both gold + corpus)
            f'/kaggle/input/swarabyanjan/{fname}',
            f'/kaggle/input/{fname}',
        ]
        for c in candidates:
            if os.path.isfile(c):
                return c
        matches = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        if matches:
            return matches[0]
        # Local paths (for development / testing)
        for local in [f'./{fname}', f'../data/{fname}', f'./data/{fname}',
                      f'/home/z/my-project/analysis/github_repo/data/{fname}',
                      f'/home/z/my-project/upload/{fname}',
                      f'/home/z/my-project/download/{fname}']:
            if os.path.isfile(local):
                return local
    return filename  # Return the primary filename if not found (will fail later with helpful error)

GOLD_PATH = find_file(GOLD_FILENAME, GOLD_FILENAME_LEGACY)
CORPUS_PATH = find_file(CORPUS_FILENAME, CORPUS_FILENAME_LEGACY)
CORPUS_METADATA_PATH = find_file(CORPUS_METADATA_FILENAME)  # Optional — None if not found
# find_file() returns the bare filename when nothing is found; normalize to None
# so downstream code can check `if CORPUS_METADATA_PATH and os.path.isfile(...)`.
if CORPUS_METADATA_PATH and not os.path.isfile(CORPUS_METADATA_PATH):
    CORPUS_METADATA_PATH = None

# Detect which version we're using (for logging + provenance in results JSON)
using_cleaned = ('Gold_Balanced_766' in GOLD_PATH) or ('Corpus_5000' in CORPUS_PATH)

OUTPUT_DIR = Path('/kaggle/working') if os.path.exists('/kaggle/working') else Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Gold standard:    {GOLD_PATH}')
print(f'Corpus:           {CORPUS_PATH}')
print(f'Corpus metadata:  {CORPUS_METADATA_PATH if CORPUS_METADATA_PATH else "(not found — optional)"}')
print(f'Output dir:       {OUTPUT_DIR}')
print(f'Using cleaned CSVs: {using_cleaned}')
if not using_cleaned:
    print('⚠️  Using legacy CSVs. For best results (Q1 publication), use the cleaned CSVs from Task 8.')
    print('   See data/README.md for the cleaned filenames and the column-rename mapping.')


Gold standard:    /kaggle/input/datasets/smalakarishere/swarabyanjan/Swarabyanjan_Gold_Balanced_766.csv
Corpus:           /kaggle/input/datasets/swagotammalakar/v18-human-gold-final/Swarabyanjan_Corpus_5000.csv
Corpus metadata:  /kaggle/input/datasets/swagotammalakar/v18-human-gold-final/Swarabyanjan_Corpus_Metadata_5000.csv
Output dir:       /kaggle/working
Using cleaned CSVs: True


## 1. SMI Criteria Scoring Functions

Eight linguistic criteria are computed for each article:

| Criterion | Description | Formula |
|-----------|-------------|---------|
| $C_1$ | Sensational Headline | Lexicon match + punctuation density |
| $C_2$ | Clickbait | Phrase match + listicle + trailing question |
| $C_3$ | Emotional Arousal | $1 - \exp(-D/\gamma)$, density per 100 words |
| $C_4$ | Attribution Gap | $1 - \lambda \cdot n_{\text{attr}} - \text{credits}$ |
| $C_5$ | Speculation | $1 - \exp(-D/\gamma)$, speculation term density |
| $C_6$ | Entertainment Displacement | Lexicon match + headline bonus |
| $C_7$ | Headline-Body Coherence | $1 - |\mathcal{T}(h) \cap \mathcal{T}(b)| / |\mathcal{T}(h)|$ |
| $C_8$ | Sensitive Topic | Coverage of communal, religious, gender, ethnic, or politically provocative themes |


In [2]:
# === SMI CRITERIA SCORING FUNCTIONS ===
# These implement the mathematical definitions C1-C7 from the paper.

import re
import math
import unicodedata

# --- Lexicons ---

SENSATIONAL_HEADLINE_TERMS = [
    "অবিশ্বাস্য", "অকল্পনীয়", "চমকে", "চাঞ্চল্যকর", "রোমহর্ষক",
    "ভয়ঙ্কর", "নারকীয়", "মর্মান্তিক", "বিভীষিকাময়",
    "চরম", "মহা", "প্রচণ্ড", "কেলেঙ্কারি", "কেলো", "হয়রানি",
    "আলোচিত", "বিতর্কিত", "রহস্যময়", "রহস্য",
    "তবে কি", "তবে কী", "কী ঘটল", "কী হলো",
    "রহস্যের", "রহস্য জট", "জট খুলল", "পর্দা ফাঁক",
    "অবাক", "হতবাক", "স্তব্ধ", "বিস্ময়ে হতবাক",
    "কাঁদছে", "ফাটল", "ছিন্নভিন্ন", "তোলপাড়", "নড়েচড়ে",
    "চাঞ্চল্য", "শিহরণ", "আঁতকে", "কাঁপিয়ে", "কাঁপছে",
]

CLICKBAIT_PHRASES = [
    "তবে কি", "তবে কী", "জানলে অবাক", "যা ঘটল", "যা কেউ বলেনি",
    "ভাবেননি", "অবাক করবে", "চমকে দেওয়া", "অজানা সত্য",
    "এক চমকে", "হয়তো ভাবেননি", "যা দেখলে", "বিশ্বাস করবেন না",
    "নিজের চোখে দেখুন", "ভিডিওতে দেখুন", "ছবিতে দেখুন",
    "পুরো ঘটনা", "পুরো রহস্য", "না জানলে মিস", "অপেক্ষা করুন",
    "রহস্যের জট", "মজার", "মজার তথ্য",
    "যা আপনি জানেন না", "গোপন তথ্য", "আসল সত্য",
    "চমকপ্রদ", "নজরকাড়া", "অভাবনীয়",
    "অবশ্যই দেখুন", "শেয়ার করুন", "ভাইরাল",
    "দেখে নিন", "জেনে নিন", "চিনে নিন",
    "বিস্ময়কর", "অকল্পনীয়", "অবিশ্বাস্য",
]

CLICKBAIT_LISTICLE_RE = re.compile(
    r"(\d+|১|২|৩|৪|৫|৬|৭|৮|৯|১০)\s*(টি|টা|ভাবে|কারণে|টিপস|পদ্ধতি|উপায়)"
)

EMOTIONAL_TERMS = [
    "অশ্রু", "কান্না", "হাহাকার", "বিলাপ", "করুণ", "করুণতা",
    "কান্নায় ভেঙে", "শোকে", "শোকাহত", "বিলাপ করছেন",
    "করুণ আর্তনাদ", "আর্তনাদ", "হাহাকার শুরু",
    "বুক ফেটে", "হৃদয় বিদারণ", "মর্মান্তিক", "নারকীয়",
    "বিভীষিকাময়", "রোমহর্ষক", "কম্পিত", "কাঁপছে",
    "হাহাকারে", "হাহাকার উঠেছে", "রোদন",
    "বিষণ্ণ", "হতাশ", "হতাশা", "নিরাশা",
    "উল্লাসে", "উল্লাসিত", "আনন্দে", "আনন্দঘন",
    "ক্ষোভে", "ক্ষুব্ধ", "রুষ্ট", "ক্ষোভ প্রকাশ",
    "বিক্ষোভ", "ধিক্কার", "নিন্দা", "প্রতিবাদ",
]

ATTRIBUTION_TERMS = [
    "বলেন", "জানিয়েছেন", "জানান", "বলা হয়েছে", "বলেছেন",
    "মতে", "অনুসারে", "সূত্রে", "সূত্র বলছে",
    "নিশ্চিত করেছেন", "নিশ্চিত করা হয়েছে",
    "প্রকাশ করেছেন", "প্রকাশ করেছে",
    "জানিয়েছে", "বলা হয়", "যোগ করেছেন",
    "রইটার্স", "রয়টার্স", "রয়টার", "বিডিনিউজ", "বাসস", "ইউএনবি",
    "এএফপি", "এপি", "ডিপিএ",
    "প্রতিবেদক", "প্রতিনিধি", "নিজস্ব প্রতিবেদক",
    "সংস্থা", "সংস্দা", "সংবাদ সংস্থা",
    "বিবৃতি", "প্রেস বিবৃতি", "বিজ্ঞপ্তি", "প্রেস রিলিজ",
    "আদালত", "পুলিশ", "মন্ত্রণালয়", "সরকার", "সংসদ",
    "বিভাগ", "অধিদপ্তর", "পরিষদ", "কমিটি", "কমিশন",
    "টিআইবি", "ট্রান্সপারেন্সি ইন্টারন্যাশনাল",
    "রিপোর্ট", "প্রতিবেদন", "তদন্ত", "অনুসন্ধান",
    "বিশেষজ্ঞ", "বিশ্লেষক", "অধ্যাপক", "ডাক্তার",
    "মামলা", "রায়", "আদেশ", "নোটিশ",
]

SPECULATION_TERMS = [
    "হতে পারে", "হতে পারেন", "থাকতে পারে", "হয়তো", "সম্ভবত",
    "মনে হচ্ছে", "মনে হয়", "অনুমান", "গুঞ্জন", "গুঞ্জন রটে",
    "সম্ভাবনা", "সম্ভব", "সম্ভাব্য",
    "জল্পনা", "কল্পনা", "জল্পনা-কল্পনা",
    "নাকি", "কি তবে", "তবে কি", "তবে কী",
    "শোনা যাচ্ছে", "জানা গেছে যে", "খবর রটে",
    "চর্চা শুরু", "বিতর্ক শুরু", "প্রশ্ন উঠেছে",
]

ENTERTAINMENT_TERMS = [
    "অভিনেত্রী", "অভিনেতা", "মডেল", "গায়ক", "গায়িকা", "নায়ক", "নায়িকা",
    "বলিউড", "হলিউড", "টলিউড", "ঢালিউড",
    "ব্যক্তিগত জীবন", "প্রেম", "প্রেমের", "বিবাহবিচ্ছেদ",
    "ছাড়াছাড়ি", "বিয়ে", "বিয়ের", "প্রেমের গল্প", "নতুন জুটি",
    "ভাইরাল", "টুইট", "ইনস্টাগ্রামে",
    "ছবি ভাইরাল", "ভিডিও ভাইরাল", "ছবি ফাঁস", "অন্তরঙ্গ",
    "চলচ্চিত্র", "প্রিমিয়ার", "শুটিং", "সিনেমা", "নাটক",
    "অভিনয়", "মুক্তি", "বক্স অফিস", "ট্রেইলর",
    "গসিপ", "ফটোশুট", "মেকআপ", "ড্রেস", "গাউন",
    "বিউটি", "ফিটনেস", "ওজন কমানো", "ফিগার", "সাইজ জিরো",
    "পুরস্কার", "এওয়ার্ড", "অস্কার",
]

SENSITIVE_TOPIC_TERMS = [
    # Communal / religious
    "মুসলমান", "হিন্দু", "ইসলাম", "হিন্দুধর্ম", "মন্দির", "মসজিদ", "মাদ্রাসা",
    "ধর্মীয়", "ধর্ম", "সাম্প্রদায়িক", "সম্প্রদায়িক", "দাঙ্গা", "দাঙ্গাহাঙ্গামা",
    "উসকানি", "উসকানি দিয়েছে", "ধর্মান্ধ", "কট্টর", "অমুসলিম", "কাফির",
    # Gender / sexual
    "ধর্ষণ", "ধর্ষিতা", "নারী নির্যাতন", "যৌন হয়রানি", "ইভ টিজিং",
    "নারীবাদী", "মেয়েদের", "নারীদের অধিকার",
    # Ethnicity / regional
    "উপজাতি", "চাকমা", "মারমা", "ত্রিপুরা", "গারো", "সাঁওতাল",
    "আদিবাসী", "পাহাড়ি", "সমতট",
    # Political provocation
    "সরকারবিরোধী", "বিরোধীদল", "ক্ষমতাসীন", "আওয়ামী লীগ", "বিএনপি",
    "জামায়াত", "জাতীয় পার্টি", "হেফাজত", "ছাত্রলীগ", "ছাত্রদল",
    "জিহাদ", "শহীদ", "শহীদের", "রাজাকার", "আলবদর",
    "বয়কট", "অবরোধ", "অচলাবস্থা", "ধর্মঘট",
    "বিচ্ছিন্নতাবাদী", "স্বাধীনতাবিরোধী",
]

BENGALI_STOPWORDS = {
    "এবং", "ও", "এর", "কে", "কেও", "তিনি", "তার", "তাকে", "তাদের",
    "এই", "সেই", "ঐ", "এক", "একটি", "একটা", "একজন",
    "হয়েছে", "হয়েছিল", "হবে", "হতে", "করেছেন", "করেছে",
    "বলেন", "বলেছেন", "যিনি", "যে", "যা",
    "আজ", "গতকাল", "আগামীকাল",
    "তবে", "কিন্তু", "আর", "অথচ", "যদিও",
    "কারণ", "তাই", "সুতরাং",
    "নিয়ে", "দিয়ে", "থেকে", "ভিতরে", "বাইরে",
    "সাথে", "সঙ্গে", "নিচে", "উপরে",
    "সব", "অনেক", "কিছু", "কোনো", "অন্য", "নিজে",
}

DATELINE_RE = re.compile(
    r"^[^\s,]{2,15}\s*,\s*[\d০-৯]|^[^\s]{2,15}\s*\([^)]+\)\s*[-—]"
)

# --- Helper functions ---

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200d", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def count_term_hits(text, terms):
    if not text:
        return 0
    return sum(1 for t in terms if t in text)

def count_total_term_hits(text, terms):
    if not text:
        return 0
    return sum(text.count(t) for t in terms)

def word_count(text):
    if not text:
        return 0
    return len(text.split())

def has_strong_attribution(headline, body):
    full = normalize_text(headline or "") + " " + normalize_text(body or "")
    credible_sources = [
        "টিআইবি", "ট্রান্সপারেন্সি", "রয়টার্স", "রইটার্স", "বিডিনিউজ",
        "বাসস", "ইউএনবি", "এএফপি", "বিশ্বব্যাংক", "আইএমএফ",
        "জাতিসংঘ", "ইউনিসেফ", "বিশ্ববিদ্যালয়", "গবেষণা", "সমীক্ষা",
        "আদালত", "পুলিশ", "র‌্যাব", "সিআইডি", "মন্ত্রণালয়",
        "প্রতিবেদক", "প্রতিনিধি", "নিজস্ব প্রতিবেদক",
        "বিজ্ঞপ্তি", "বিবৃতি",
    ]
    return any(src in full for src in credible_sources)

def has_dateline(body):
    b = normalize_text(body or "")[:200]
    return bool(DATELINE_RE.match(b))

# --- Seven Criteria Scoring Functions ---

def C1_sensational_headline(headline):
    """C1: Sensational headline score in [0,1]."""
    if not headline:
        return 0.0
    h = normalize_text(headline)
    hits = count_term_hits(h, SENSATIONAL_HEADLINE_TERMS)
    marks = h.count("!") + h.count("?")
    base = min(hits / 2.0, 1.0)
    mark_bonus = min(marks / 1.5, 0.3)
    return min(base + mark_bonus, 1.0)

def C2_clickbait(headline, body):
    """C2: Clickbait score in [0,1]."""
    h = normalize_text(headline or "")
    phrase_hits = count_term_hits(h, CLICKBAIT_PHRASES)
    listicle_hit = 1 if CLICKBAIT_LISTICLE_RE.search(h) else 0
    trailing_q = 1 if (h.endswith("?") or h.endswith("…") or h.endswith("...")) else 0
    base = min(phrase_hits / 1.5, 1.0)
    bonus = 0.15 * listicle_hit + 0.20 * trailing_q
    return min(base + bonus, 1.0)

def C3_emotional(body):
    """C3: Emotional arousal score in [0,1].
    Formula: 1 - exp(-D/gamma), D = density per 100 words
    """
    b = normalize_text(body or "")
    if not b:
        return 0.0
    wc = word_count(b)
    if wc == 0:
        return 0.0
    hits = count_total_term_hits(b, EMOTIONAL_TERMS)
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.2
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def C4_attribution_gap(headline, body):
    """C4: Attribution gap score in [0,1].
    Formula: max(1 - lambda*n_attr - credits, 0) + short_penalty
    """
    b = normalize_text(body or "")
    h = normalize_text(headline or "")
    full = h + " " + b
    wc = word_count(b)
    if wc == 0:
        return 1.0
    attr_hits = count_term_hits(full, ATTRIBUTION_TERMS)
    has_strong = has_strong_attribution(h, b)
    has_dl = has_dateline(b)
    lam = 0.10
    base = max(1.0 - lam * attr_hits, 0.0)
    if has_strong:
        base = max(base - 0.30, 0.0)
    if has_dl:
        base = max(base - 0.15, 0.0)
    if wc < 100:
        base = min(base + 0.05, 1.0)
    return min(max(base, 0.0), 1.0)

def C5_speculation(body):
    """C5: Speculation-as-fact score in [0,1].
    Formula: 1 - exp(-D/gamma)
    """
    b = normalize_text(body or "")
    if not b:
        return 0.0
    wc = word_count(b)
    hits = count_total_term_hits(b, SPECULATION_TERMS)
    if wc == 0:
        return 0.0
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.2
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def C6_entertainment(headline, body):
    """C6: Entertainment displacement score in [0,1].
    Formula: min(hits/alpha + 0.25*headline_hits, 1)
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    full = h + " " + b
    hits = count_term_hits(full, ENTERTAINMENT_TERMS)
    headline_hits = count_term_hits(h, ENTERTAINMENT_TERMS)
    alpha = 3.0
    base = min(hits / alpha, 1.0)
    headline_bonus = min(0.25 * headline_hits, 0.5)
    return min(base + headline_bonus, 1.0)

def C7_coherence(headline, body):
    """C7: Headline-body coherence (mismatch) score in [0,1].
    Formula: 1 - overlap_ratio if overlap < tau, else 0
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    if not h or not b:
        return 0.3
    h_tokens = set(re.findall(r"[\u0980-\u09FF]+|[A-Za-z]+|\d+", h))
    b_tokens = set(re.findall(r"[\u0980-\u09FF]+|[A-Za-z]+|\d+", b))
    h_tokens = {t for t in h_tokens if len(t) > 1 and t not in BENGALI_STOPWORDS}
    b_tokens = {t for t in b_tokens if len(t) > 1 and t not in BENGALI_STOPWORDS}
    if not h_tokens:
        return 0.3
    overlap = h_tokens & b_tokens
    overlap_ratio = len(overlap) / len(h_tokens)
    tau = 0.35
    if overlap_ratio < tau:
        return 1.0 - overlap_ratio
    return 0.0

def C8_sensitive_topic(headline, body):
    """C8: Sensitive topic score in [0,1].
    Formula: 1 - exp(-D/gamma), D = density per 100 words on
    combined headline+body, gamma = 1.5.
    """
    h = normalize_text(headline or "")
    b = normalize_text(body or "")
    full = h + " " + b
    if not full.strip():
        return 0.0
    wc = word_count(full)
    if wc == 0:
        return 0.0
    hits = count_total_term_hits(full, SENSITIVE_TOPIC_TERMS)
    density = hits / max(wc / 100.0, 1.0)
    gamma = 1.5
    score = 1 - math.exp(-density / gamma)
    return min(score, 1.0)

def compute_all_criteria(headline, body):
    """Compute all 8 criteria scores for an article."""
    return {
        'C1': round(C1_sensational_headline(headline), 4),
        'C2': round(C2_clickbait(headline, body), 4),
        'C3': round(C3_emotional(body), 4),
        'C4': round(C4_attribution_gap(headline, body), 4),
        'C5': round(C5_speculation(body), 4),
        'C6': round(C6_entertainment(headline, body), 4),
        'C7': round(C7_coherence(headline, body), 4),
        'C8': round(C8_sensitive_topic(headline, body), 4),
    }

print('SMI criteria scoring functions defined.')
print(f'  C1: Sensational Headline (lexicon size: {len(SENSATIONAL_HEADLINE_TERMS)})')
print(f'  C2: Clickbait (lexicon size: {len(CLICKBAIT_PHRASES)})')
print(f'  C3: Emotional Arousal (lexicon size: {len(EMOTIONAL_TERMS)})')
print(f'  C4: Attribution Gap (lexicon size: {len(ATTRIBUTION_TERMS)})')
print(f'  C5: Speculation (lexicon size: {len(SPECULATION_TERMS)})')
print(f'  C6: Entertainment (lexicon size: {len(ENTERTAINMENT_TERMS)})')
print(f'  C7: Headline-Body Coherence')
print(f'  C8: Sensitive Topic (lexicon size: {len(SENSITIVE_TOPIC_TERMS)})')


SMI criteria scoring functions defined.
  C1: Sensational Headline (lexicon size: 41)
  C2: Clickbait (lexicon size: 38)
  C3: Emotional Arousal (lexicon size: 40)
  C4: Attribution Gap (lexicon size: 59)
  C5: Speculation (lexicon size: 26)
  C6: Entertainment (lexicon size: 49)
  C7: Headline-Body Coherence
  C8: Sensitive Topic (lexicon size: 57)


## 2. Load Human-Verified Gold Standard

The gold standard consists of 766 articles (383 yellow + 383 non-yellow) annotated using the 8-criteria protocol with deep reasoning.


In [3]:
# === Load gold standard ===
# Task 9 (2026-07-19): Auto-detect column names (handles both legacy and
# cleaned CSV schemas) and handle stub articles (body_text="not_available")
# gracefully by replacing the placeholder with an empty string for SMI
# density-based scoring (C3/C5/C6 return 0.0 for empty bodies).

gold = pd.read_csv(GOLD_PATH)
print(f'Gold loaded: {gold.shape}')
print(f'Columns: {list(gold.columns)}')

# Column auto-detection — handles both legacy and cleaned schemas
def detect_gold_columns(df):
    """Detect column names, handling both legacy and cleaned schemas.

    Legacy gold CSV (Swarabyanjan_BEST_BALANCED_1to1.csv):
      article_id, headline, body_text, news_source, article_length,
      best_label, best_confidence, best_note

    Cleaned gold CSV (Swarabyanjan_Gold_Balanced_766.csv):
      article_id, headline, body_text, corpus_batch, article_length,
      best_label, best_confidence, best_note

    Only the source column name differs (news_source vs corpus_batch).
    """
    cols = {}

    # article_id — same in both schemas
    if 'article_id' not in df.columns:
        raise ValueError(f'No article_id column found. Available: {list(df.columns)}')
    cols['id'] = 'article_id'

    # headline — same in both
    cols['headline'] = 'headline'

    # body_text — same in both
    cols['body'] = 'body_text'

    # corpus_batch (cleaned) vs news_source (legacy)
    if 'corpus_batch' in df.columns:
        cols['source'] = 'corpus_batch'
    elif 'news_source' in df.columns:
        cols['source'] = 'news_source'
    else:
        raise ValueError(f'No corpus_batch/news_source column found. Available: {list(df.columns)}')

    # article_length — same in both
    cols['length'] = 'article_length'

    # best_label — same in both
    cols['label'] = 'best_label'

    # best_confidence — same in both
    cols['confidence'] = 'best_confidence'

    # best_note — same in both
    cols['note'] = 'best_note'

    return cols

gold_cols = detect_gold_columns(gold)
print(f'Detected columns: {gold_cols}')

# Normalize: ensure both corpus_batch and news_source columns exist (one may
# be missing depending on schema; downstream cells use either name).
source_col = gold_cols['source']
if source_col == 'news_source' and 'corpus_batch' not in gold.columns:
    gold['corpus_batch'] = gold['news_source']
elif source_col == 'corpus_batch' and 'news_source' not in gold.columns:
    gold['news_source'] = gold['corpus_batch']

# Ensure text columns are strings (handle NaN)
gold['headline'] = gold['headline'].fillna('').astype(str)
gold['body_text'] = gold['body_text'].fillna('').astype(str)

# Document stub articles (body_text == "not_available")
n_stub_gold = int((gold['body_text'] == 'not_available').sum())
if n_stub_gold > 0:
    print(f'\n⚠️  {n_stub_gold} stub articles found in gold (body_text = "not_available").')
    print(f'   These are legitimate "stub article" cases labeled yellow via C7 (headline-body mismatch).')
    print(f'   SMI criteria C3/C5/C6 (density-based) will return 0 for these — handled gracefully.')

# Handle stub articles: replace "not_available" with empty string for SMI scoring.
# The original "not_available" is a placeholder, not real text — preserve it in
# body_text_original for reproducibility / audit.
gold['body_text_original'] = gold['body_text'].copy()
gold.loc[gold['body_text'] == 'not_available', 'body_text'] = ''

# Verify stub replacement
n_stub_replaced = int((gold['body_text_original'] == 'not_available').sum())
assert n_stub_replaced == n_stub_gold, f'Stub count mismatch: {n_stub_replaced} vs {n_stub_gold}'

print(f'\nGold standard: {len(gold)} articles')
print(f'Yellow: {gold["best_label"].sum()}')
print(f'Non-yellow: {(gold["best_label"]==0).sum()}')
print(f'\nLabel distribution: {gold["best_label"].value_counts().to_dict()}')
if 'best_confidence' in gold.columns:
    print(f'Confidence distribution: {gold["best_confidence"].value_counts().to_dict()}')
print(f'\nColumns: {list(gold.columns)}')
print(f'Stub articles replaced with empty string: {n_stub_gold} in gold')
gold.head(3)


Gold loaded: (766, 8)
Columns: ['article_id', 'headline', 'body_text', 'corpus_batch', 'article_length', 'best_label', 'best_confidence', 'best_note']
Detected columns: {'id': 'article_id', 'headline': 'headline', 'body': 'body_text', 'source': 'corpus_batch', 'length': 'article_length', 'label': 'best_label', 'confidence': 'best_confidence', 'note': 'best_note'}

⚠️  2 stub articles found in gold (body_text = "not_available").
   These are legitimate "stub article" cases labeled yellow via C7 (headline-body mismatch).
   SMI criteria C3/C5/C6 (density-based) will return 0 for these — handled gracefully.

Gold standard: 766 articles
Yellow: 383
Non-yellow: 383

Label distribution: {0: 383, 1: 383}
Confidence distribution: {'H': 376, 'M': 328, 'L': 62}

Columns: ['article_id', 'headline', 'body_text', 'corpus_batch', 'article_length', 'best_label', 'best_confidence', 'best_note', 'news_source', 'body_text_original']
Stub articles replaced with empty string: 2 in gold


,article_id,headline,body_text,corpus_batch,article_length,best_label,best_confidence,best_note,news_source,body_text_original
0,v18_2830,কুমিল্লায় ‘ডাকাতের গুলিতে ডাকাত’ নিহত,"দাউদকান্দিথানার ওসি মিজানুর রহমান বলেন, “দুই দ...",corpus_expansion_5000,694,0,H,Crime news with named Daudkandi OC Mizanur Rah...,corpus_expansion_5000,"দাউদকান্দিথানার ওসি মিজানুর রহমান বলেন, “দুই দ..."
1,v18_0000,২০১৮ সালে বিশ্বজুড়ে ৯৭ সাংবাদিক খুন,বিশ্বজুড়ে সাংবাদিকদের ওপর হামলার ঘটনাগুলোতে ব...,new_low_w0_pany,1205,0,H,Factual international press-freedom report wit...,new_low_w0_pany,বিশ্বজুড়ে সাংবাদিকদের ওপর হামলার ঘটনাগুলোতে ব...
2,v18_4780,বিশ্বকাপে আফগানদের বিপক্ষে টেস্ট খেললেন ধোনি?,খেলা শুরুর আগেই 'ফেভারিট বনাম লাস্টবয়' শিরোনা...,corpus_expansion_5000,967,1,M,[R1-BROADER] Clickbait sports headline ('?'); ...,corpus_expansion_5000,খেলা শুরুর আগেই 'ফেভারিট বনাম লাস্টবয়' শিরোনা...


## 3. Compute SMI Criteria Scores for Gold Standard

For each article in the gold standard, we compute the 8 criteria scores $\mathbf{C}(a) = (C_1(a), C_2(a), \ldots, C_8(a))^T$.


In [4]:
# === Compute criteria scores for gold standard ===
print('Computing SMI criteria scores for 766 gold-standard articles...')
t0 = time.time()

criteria_rows = []
for _, row in gold.iterrows():
    c = compute_all_criteria(row['headline'], row['body_text'])
    c['article_id'] = row['article_id']
    c['true_label'] = int(row['best_label'])
    criteria_rows.append(c)

gold_criteria = pd.DataFrame(criteria_rows)
t1 = time.time()
print(f'Done in {t1-t0:.2f}s')
print(f'Shape: {gold_criteria.shape}')

# Show criteria score statistics
print('\nCriteria score means by label:')
for c in ['C1','C2','C3','C4','C5','C6','C7','C8']:
    m_y = gold_criteria[gold_criteria.true_label==1][c].mean()
    m_n = gold_criteria[gold_criteria.true_label==0][c].mean()
    print(f'  {c}: Yellow={m_y:.3f}, Non-yellow={m_n:.3f}, Diff={m_y-m_n:+.3f}')

gold_criteria.head()

Computing SMI criteria scores for 766 gold-standard articles...
Done in 2.52s
Shape: (766, 10)

Criteria score means by label:
  C1: Yellow=0.186, Non-yellow=0.016, Diff=+0.171
  C2: Yellow=0.029, Non-yellow=0.002, Diff=+0.027
  C3: Yellow=0.060, Non-yellow=0.050, Diff=+0.010
  C4: Yellow=0.598, Non-yellow=0.361, Diff=+0.237
  C5: Yellow=0.202, Non-yellow=0.101, Diff=+0.102
  C6: Yellow=0.381, Non-yellow=0.092, Diff=+0.289
  C7: Yellow=0.136, Non-yellow=0.092, Diff=+0.044
  C8: Yellow=0.167, Non-yellow=0.239, Diff=-0.072


,C1,C2,C3,C4,C5,C6,C7,C8,article_id,true_label
0,0.0,0.0,0.0000,0.25,0.0,0.0000,0.0,0.9619,v18_2830,0
1,0.0,0.0,0.0000,0.20,0.0,0.0000,0.0,0.0000,v18_0000,0
2,0.3,0.2,0.0000,1.00,0.0,0.3333,0.0,0.0000,v18_4780,1
3,0.0,0.0,0.0000,0.20,0.0,1.0000,0.0,0.6832,v18_0416,0
4,0.0,0.0,0.4606,0.90,0.0,0.0000,0.0,0.0000,v18_4000,1


## 4. Learn SMI Weights via Logistic Regression

The weight vector $\mathbf{w}$ and bias $w_0$ are learned by minimizing the $\ell_2$-regularized logistic loss:

$$\min_{\mathbf{w}, w_0} \frac{1}{n} \sum_{j=1}^{n} \mathcal{L}_{\text{log}}(y_j, \sigma(\mathbf{w}^T \mathbf{C}(a_j) + w_0)) + \lambda \|\mathbf{w}\|_2^2$$

We use 5-fold stratified cross-validation to evaluate the learned weights.


In [5]:
# === Learn SMI weights ===
X = gold_criteria[['C1','C2','C3','C4','C5','C6','C7','C8']].values
y = gold_criteria['true_label'].values

# 5-fold CV to evaluate
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y))

cv_f1s = []
cv_accs = []
all_preds = np.zeros(len(y), dtype=int)
all_probs = np.full(len(y), np.nan)

for fold_i, (train_idx, val_idx) in enumerate(folds):
    lr = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', random_state=SEED)
    lr.fit(X[train_idx], y[train_idx])
    preds = lr.predict(X[val_idx])
    probs = lr.predict_proba(X[val_idx])[:, 1]
    all_preds[val_idx] = preds
    all_probs[val_idx] = probs
    f1 = f1_score(y[val_idx], preds)
    acc = accuracy_score(y[val_idx], preds)
    cv_f1s.append(f1)
    cv_accs.append(acc)
    print(f'  Fold {fold_i+1}: F1={f1:.4f}, Acc={acc:.4f}')

print(f'\nSMI 5-Fold CV Results:')
print(f'  F1:        {np.mean(cv_f1s):.4f} ± {np.std(cv_f1s):.4f}')
print(f'  Accuracy:  {np.mean(cv_accs):.4f} ± {np.std(cv_accs):.4f}')

# Overall metrics (aggregated predictions)
print(f'\nSMI Aggregated Metrics (all 766 predictions):')
print(f'  F1:        {f1_score(y, all_preds):.4f}')
print(f'  Accuracy:  {accuracy_score(y, all_preds):.4f}')
print(f'  Precision: {precision_score(y, all_preds):.4f}')
print(f'  Recall:    {recall_score(y, all_preds):.4f}')
print(f'  Kappa:     {cohen_kappa_score(y, all_preds):.4f}')
print(f'  MCC:       {matthews_corrcoef(y, all_preds):.4f}')
print(f'  AUC:       {roc_auc_score(y, all_probs):.4f}')

  Fold 1: F1=0.7361, Acc=0.7532
  Fold 2: F1=0.9079, Acc=0.9085
  Fold 3: F1=0.8163, Acc=0.8235
  Fold 4: F1=0.7826, Acc=0.8039
  Fold 5: F1=0.8027, Acc=0.8105

SMI 5-Fold CV Results:
  F1:        0.8091 ± 0.0564
  Accuracy:  0.8199 ± 0.0503

SMI Aggregated Metrics (all 766 predictions):
  F1:        0.8104
  Accuracy:  0.8198
  Precision: 0.8551
  Recall:    0.7702
  Kappa:     0.6397
  MCC:       0.6429
  AUC:       0.8981


In [6]:
# === Train final model on ALL 766 articles (for annotation) ===
final_lr = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', random_state=SEED)
final_lr.fit(X, y)

print('=== Learned SMI Weights ===')
criteria_names = [
    'C1: Sensational Headline',
    'C2: Clickbait',
    'C3: Emotional Arousal',
    'C4: Attribution Gap',
    'C5: Speculation',
    'C6: Entertainment Displacement',
    'C7: Headline-Body Coherence',
    'C8: Sensitive Topic',
]

for name, weight in sorted(zip(criteria_names, final_lr.coef_[0]), key=lambda x: -abs(x[1])):
    print(f'  {name:<40} w = {weight:+.4f}')
print(f'  {"Bias (w_0)":<40} w_0 = {final_lr.intercept_[0]:+.4f}')

# Find optimal threshold
from sklearn.metrics import precision_recall_curve
probs_all = final_lr.predict_proba(X)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y, probs_all)
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
best_idx = np.argmax(f1s)
tau_star = thresholds[best_idx]
print(f'\nOptimal threshold: τ* = {tau_star:.4f}')
print(f'  At τ*: F1={f1s[best_idx]:.4f}, Precision={precisions[best_idx]:.4f}, Recall={recalls[best_idx]:.4f}')

=== Learned SMI Weights ===
  C1: Sensational Headline                 w = +6.6100
  C6: Entertainment Displacement           w = +2.2137
  C4: Attribution Gap                      w = +2.0237
  C2: Clickbait                            w = +1.1706
  C5: Speculation                          w = +1.0843
  C7: Headline-Body Coherence              w = +0.6151
  C3: Emotional Arousal                    w = +0.4006
  C8: Sensitive Topic                      w = -0.0755
  Bias (w_0)                               w_0 = -2.2595

Optimal threshold: τ* = 0.4168
  At τ*: F1=0.8506, Precision=0.8256, Recall=0.8773


## 5. SMI Automated Annotation — Apply to 5,000 Articles

Using the learned weights $\mathbf{w}$ and threshold $\tau^*$, we annotate all 5,000 articles in the corpus:

$$\hat{y}(a) = \begin{cases} 1 & \text{if } \sigma(\mathbf{w}^T \mathbf{C}(a) + w_0) \geq \tau^* \\ 0 & \text{otherwise} \end{cases}$$


In [7]:
# === Load full 5,000-article corpus ===
# Task 5 (2026-07-19): Map legacy column names to the canonical schema the SMI
# scoring functions expect.
# Task 9 (2026-07-19): Now supports BOTH the cleaned CSV (Swarabyanjan_Corpus_5000.csv)
# with canonical column names AND the legacy CSV (v18_HUMAN_GOLD_FINAL_v20_GOLD_5000.csv)
# via automatic schema detection. Also handles stub articles (body_text="not_available")
# gracefully by replacing the placeholder with an empty string for SMI scoring.

corpus = pd.read_csv(CORPUS_PATH)
print(f'Corpus loaded: {corpus.shape}')
print(f'Columns: {list(corpus.columns)}')

# Column mapping — handles both legacy (v18_HUMAN_GOLD_FINAL_v20_GOLD_5000.csv)
# and cleaned (Swarabyanjan_Corpus_5000.csv) schemas.
if 'article_id' in corpus.columns:
    # Cleaned schema — columns already have the canonical names
    print('Using cleaned corpus schema (Swarabyanjan_Corpus_5000.csv)')
else:
    # Legacy schema — apply column mapping
    print('Using legacy corpus schema (v18_HUMAN_GOLD_FINAL_v20_GOLD_5000.csv)')
    COLUMN_MAP = {
        'v18_id':         'article_id',
        'body_preview':   'body_text',
        'body_full_len':  'article_length',
        'source':         'corpus_batch',
        'MY_LABEL':       'human_label',
        'MY_CONFIDENCE':  'human_confidence',
        'MY_NOTE':        'annotation_provenance',
    }
    rename_dict = {k: v for k, v in COLUMN_MAP.items() if k in corpus.columns}
    corpus = corpus.rename(columns=rename_dict)
    print(f'Renamed columns: {rename_dict}')

# Ensure required columns exist
required = ['article_id', 'headline', 'body_text']
missing = [c for c in required if c not in corpus.columns]
if missing:
    raise ValueError(f'Corpus CSV missing required columns after mapping: {missing}. '
                     f'Available: {list(corpus.columns)}')

# Normalize: ensure corpus_batch exists (cleaned schema has it; legacy schema
# gets it via the rename above). Also create a news_source alias for any
# downstream code that references the legacy column name.
if 'corpus_batch' not in corpus.columns and 'news_source' in corpus.columns:
    corpus['corpus_batch'] = corpus['news_source']
if 'news_source' not in corpus.columns and 'corpus_batch' in corpus.columns:
    corpus['news_source'] = corpus['corpus_batch']

# Handle missing body_text / headline (some rows may have NaN)
corpus['body_text'] = corpus['body_text'].fillna('').astype(str)
corpus['headline'] = corpus['headline'].fillna('').astype(str)

# Document the body_preview truncation issue
# (body_preview is capped at 1200 chars when body_full_len > 1200)
corpus_n_truncated = 0
corpus_pct_truncated = 0.0
if 'article_length' in corpus.columns:
    corpus_n_truncated = int((corpus['body_text'].str.len() < corpus['article_length']).sum())
    corpus_pct_truncated = float(100.0 * corpus_n_truncated / len(corpus))
    print(f'\n⚠️  Body text truncation notice:')
    print(f'   {corpus_n_truncated} / {len(corpus)} articles ({corpus_pct_truncated:.1f}%) have truncated body text.')
    print(f'   The source CSV stores body_preview which is capped at 1200 characters')
    print(f'   when body_full_len > 1200. SMI criteria C3 (Emotional), C5 (Speculation),')
    print(f'   C6 (Entertainment) use word-density formulas that may be slightly affected.')
    print(f'   This is documented in the notebook and in smi_annotation_results.json.')

# Document stub articles (body_text == "not_available")
n_stub_corpus = int((corpus['body_text'] == 'not_available').sum())
if n_stub_corpus > 0:
    print(f'\n⚠️  {n_stub_corpus} stub articles found in corpus (body_text = "not_available").')
    print(f'   SMI criteria C3/C5/C6 will return 0 for these — handled gracefully.')

# Handle stub articles: replace "not_available" with empty string for SMI scoring.
# Preserve the original placeholder in body_text_original for reproducibility.
corpus['body_text_original'] = corpus['body_text'].copy()
corpus.loc[corpus['body_text'] == 'not_available', 'body_text'] = ''
# Recount stubs from the preserved original column (should match n_stub_corpus)
n_stub_corpus = int((corpus['body_text_original'] == 'not_available').sum())

# Optionally load the corpus metadata CSV (for reproducibility — not used by SMI)
if CORPUS_METADATA_PATH and os.path.isfile(CORPUS_METADATA_PATH):
    try:
        corpus_metadata = pd.read_csv(CORPUS_METADATA_PATH)
        print(f'\nCorpus metadata loaded: {corpus_metadata.shape} (for reproducibility — not used by SMI)')
    except Exception as e:
        corpus_metadata = None
        print(f'\nCorpus metadata: failed to load ({e}) — optional, continuing')
else:
    corpus_metadata = None
    print(f'\nCorpus metadata: not loaded (optional file)')

print(f'\nCorpus after schema mapping + stub handling: {corpus.shape}')
print(f'Columns: {list(corpus.columns)}')
if 'human_label' in corpus.columns:
    print(f'human_label distribution: {corpus["human_label"].value_counts().to_dict()}')

# === Compute SMI criteria for all 5,000 articles ===
print('\nComputing SMI criteria for 5,000 articles...')
t_start = time.time()

corpus_criteria = []
for _, row in corpus.iterrows():
    c = compute_all_criteria(row['headline'], row['body_text'])
    c['article_id'] = row['article_id']
    # Preserve human annotations for the Section 6.5 out-of-sample evaluation
    if 'human_label' in row and pd.notna(row.get('human_label', None)):
        c['human_label'] = int(row['human_label'])
    if 'human_confidence' in corpus.columns:
        c['human_confidence'] = row.get('human_confidence', None)
    # corpus_batch is the canonical name; news_source is the legacy alias
    if 'corpus_batch' in row:
        c['corpus_batch'] = row['corpus_batch']
    elif 'news_source' in row:
        c['corpus_batch'] = row['news_source']
    corpus_criteria.append(c)

corpus_crit_df = pd.DataFrame(corpus_criteria)
t_end = time.time()
smi_time = t_end - t_start

print(f'Done in {smi_time:.2f}s')
print(f'Articles processed: {len(corpus_crit_df)}')
print(f'Time per article: {smi_time/len(corpus_crit_df)*1000:.2f}ms')


Corpus loaded: (5000, 8)
Columns: ['article_id', 'headline', 'body_text', 'article_length', 'corpus_batch', 'human_label', 'human_confidence', 'annotation_provenance']
Using cleaned corpus schema (Swarabyanjan_Corpus_5000.csv)

⚠️  Body text truncation notice:
   549 / 5000 articles (11.0%) have truncated body text.
   The source CSV stores body_preview which is capped at 1200 characters
   when body_full_len > 1200. SMI criteria C3 (Emotional), C5 (Speculation),
   C6 (Entertainment) use word-density formulas that may be slightly affected.
   This is documented in the notebook and in smi_annotation_results.json.

⚠️  4 stub articles found in corpus (body_text = "not_available").
   SMI criteria C3/C5/C6 will return 0 for these — handled gracefully.

Corpus metadata loaded: (5000, 9) (for reproducibility — not used by SMI)

Corpus after schema mapping + stub handling: (5000, 10)
Columns: ['article_id', 'headline', 'body_text', 'article_length', 'corpus_batch', 'human_label', 'human_con

In [8]:
# === Apply SMI annotation ===
X_corpus = corpus_crit_df[['C1','C2','C3','C4','C5','C6','C7','C8']].values

# Compute SMI composite scores
smi_scores = final_lr.predict_proba(X_corpus)[:, 1]

# Apply decision rule
smi_labels = (smi_scores >= tau_star).astype(int)

corpus_crit_df['SMI_score'] = smi_scores
corpus_crit_df['SMI_label'] = smi_labels

print('=== SMI Automated Annotation Results ===')
print(f'Total articles annotated: {len(corpus_crit_df)}')
print(f'SMI Yellow (1):     {smi_labels.sum()} ({smi_labels.mean()*100:.1f}%)')
print(f'SMI Non-yellow (0): {(1-smi_labels).sum()} ({(1-smi_labels).mean()*100:.1f}%)')
if 'human_label' in corpus_crit_df.columns:
    h = corpus_crit_df['human_label'].dropna().astype(int)
    print(f'Human Yellow (1):   {int(h.sum())} ({h.mean()*100:.1f}%) — from MY_LABEL column')
    print(f'Human Non-yellow (0): {int((1-h).sum())} ({(1-h).mean()*100:.1f}%) — from MY_LABEL column')
print(f'\nAnnotation time: {smi_time:.2f}s for 5,000 articles')
print(f'Time per article: {smi_time/5000*1000:.2f}ms')
print(f'\nFor comparison:')
print(f'  Human annotation: ~2 min/article × 5,000 = ~167 hours')
print(f'  SMI annotation:   {smi_time:.2f}s')
print(f'  Speedup: {167*3600/smi_time:.0f}x')

=== SMI Automated Annotation Results ===
Total articles annotated: 5000
SMI Yellow (1):     1289 (25.8%)
SMI Non-yellow (0): 3711 (74.2%)
Human Yellow (1):   2298 (46.0%) — from MY_LABEL column
Human Non-yellow (0): 2702 (54.0%) — from MY_LABEL column

Annotation time: 16.17s for 5,000 articles
Time per article: 3.23ms

For comparison:
  Human annotation: ~2 min/article × 5,000 = ~167 hours
  SMI annotation:   16.17s
  Speedup: 37191x


## 6. Validation: Held-Out Evaluation on Gold Standard

We use a 70/30 stratified split (536 train / 230 test) to evaluate SMI generalization. The 5-fold CV F1 from Section 4 is reported alongside for variance estimation. The SMI model trained on all 766 gold articles (cell 10) is still used to annotate the 5,000-article corpus — the held-out split only affects the evaluation claim, not the corpus annotation step.

> **Note:** This is the *in-sample* held-out evaluation (70/30 split of the 766 gold articles, where both train and test splits come from the same 766-article gold standard). For the **true out-of-sample** evaluation on the 4,234 non-gold articles of the 5,000-article corpus (articles SMI has never seen during training), see **Section 6.5** below.


In [9]:
# === Validate SMI annotation on gold standard (HELD-OUT 70/30 split) ===
# Bug-A fix: the previous version of this cell computed SMI vs Human F1
# on the same 766 articles used to train final_lr — i.e. in-sample.
# That F1=0.846 was inflated by data leakage. This cell now performs a
# proper held-out 70/30 stratified evaluation:
#   * Train a FRESH LogisticRegression on the 70% train split only
#   * Choose the decision threshold on the TRAIN split (no test leakage)
#   * Evaluate on the 30% test split
# The 5-fold CV F1 from cell 9 is the more robust generalization
# estimate; the held-out F1 here is the primary reported number.

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve

# --- Build held-out split on the 8 criteria feature matrix ---
X_full = gold_criteria[['C1','C2','C3','C4','C5','C6','C7','C8']].values
y_full = gold_criteria['true_label'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_full, y_full,
    test_size=0.30, stratify=y_full, random_state=SEED,
)
print(f'Held-out split (70/30 stratified, seed={SEED}):')
print(f'  train: {len(y_tr)} articles ({int(y_tr.sum())} yellow / {int((y_tr==0).sum())} non-yellow)')
print(f'  test:  {len(y_te)} articles ({int(y_te.sum())} yellow / {int((y_te==0).sum())} non-yellow)')

# --- Train a FRESH logistic regression on the train split only ---
heldout_lr = LogisticRegression(
    C=1.0, max_iter=2000, class_weight='balanced', random_state=SEED
)
heldout_lr.fit(X_tr, y_tr)

# --- Choose decision threshold on TRAIN split (no test leakage) ---
probs_tr = heldout_lr.predict_proba(X_tr)[:, 1]
prec_tr, rec_tr, thr_tr = precision_recall_curve(y_tr, probs_tr)
f1s_tr = 2 * prec_tr * rec_tr / (prec_tr + rec_tr + 1e-12)
tau_heldout = float(thr_tr[int(np.argmax(f1s_tr))])
print(f'Optimal threshold (chosen on TRAIN split only): τ = {tau_heldout:.4f}')

# --- Predict on the test split ---
heldout_probs = heldout_lr.predict_proba(X_te)[:, 1]
heldout_labels = (heldout_probs >= tau_heldout).astype(int)

held_out_accuracy  = accuracy_score(y_te, heldout_labels)
held_out_precision = precision_score(y_te, heldout_labels)
held_out_recall    = recall_score(y_te, heldout_labels)
held_out_f1        = f1_score(y_te, heldout_labels)
held_out_kappa     = cohen_kappa_score(y_te, heldout_labels)
held_out_mcc       = matthews_corrcoef(y_te, heldout_labels)
held_out_auc       = roc_auc_score(y_te, heldout_probs)

print(f'\n=== Held-out SMI Evaluation (70/30 split: 536 train / 230 test) ===')
print(f'{"Metric":<20} {"Value":>10}')
print('-' * 35)
print(f'{"Accuracy":<20} {held_out_accuracy:>10.4f}')
print(f'{"Precision":<20} {held_out_precision:>10.4f}')
print(f'{"Recall":<20} {held_out_recall:>10.4f}')
print(f'{"F1 Score":<20} {held_out_f1:>10.4f}')
print(f'{"Cohen Kappa":<20} {held_out_kappa:>10.4f}')
print(f'{"MCC":<20} {held_out_mcc:>10.4f}')
print(f'{"AUC":<20} {held_out_auc:>10.4f}')

# --- Confusion matrix on test split ---
cm = confusion_matrix(y_te, heldout_labels)
print(f'\nConfusion Matrix (test split):')
print(f'                  SMI=Non-yellow  SMI=Yellow')
print(f'  Human=Non-yellow     {cm[0][0]:>8}        {cm[0][1]:>8}')
print(f'  Human=Yellow         {cm[1][0]:>8}        {cm[1][1]:>8}')

print(f'\nClassification Report (test split):')
print(classification_report(y_te, heldout_labels, target_names=['Non-yellow (0)', 'Yellow (1)']))

print(f'\nNOTE: 5-fold CV F1 (cell 9) = {np.mean(cv_f1s):.4f} ± {np.std(cv_f1s):.4f} is the more robust generalization estimate.')

# --- For interpretability in cell 17 we still need SMI scores/labels on the
# full gold standard from the model trained on ALL 766 articles (final_lr).
# These are IN-SAMPLE and are used ONLY for per-article explanation plots,
# NOT as an evaluation metric. The held-out numbers above are the only
# evaluation reported.
gold_smi_scores = final_lr.predict_proba(X)[:, 1]
gold_smi_labels = (gold_smi_scores >= tau_star).astype(int)

Held-out split (70/30 stratified, seed=42):
  train: 536 articles (268 yellow / 268 non-yellow)
  test:  230 articles (115 yellow / 115 non-yellow)
Optimal threshold (chosen on TRAIN split only): τ = 0.3746

=== Held-out SMI Evaluation (70/30 split: 536 train / 230 test) ===
Metric                    Value
-----------------------------------
Accuracy                 0.8217
Precision                0.7761
Recall                   0.9043
F1 Score                 0.8353
Cohen Kappa              0.6435
MCC                      0.6524
AUC                      0.9139

Confusion Matrix (test split):
                  SMI=Non-yellow  SMI=Yellow
  Human=Non-yellow           85              30
  Human=Yellow               11             104

Classification Report (test split):
                precision    recall  f1-score   support

Non-yellow (0)       0.89      0.74      0.81       115
    Yellow (1)       0.78      0.90      0.84       115

      accuracy                           0.82       

## 6.5 True Out-of-Sample Evaluation: SMI on 4,234 Non-Gold Articles

The 5,000-article corpus contains 766 articles that overlap with the gold standard (used for SMI training) and **4,234 articles that SMI has never seen**. The corpus CSV provides a human annotation (`MY_LABEL`) for all 5,000 articles, which allows us to evaluate SMI on a true held-out set.

**Important caveat — label noise:**

The `MY_LABEL` column in the corpus is the **pre-revision** human annotation. The gold standard's `best_label` was subsequently revised from `MY_LABEL`. On the 766 overlapping articles, `best_label` and `MY_LABEL` agree only **59.9%** of the time (crosstab below). This means `MY_LABEL` is a noisier label than `best_label`, and SMI's F1 against `MY_LABEL` on the 4,234 non-gold articles is a **lower bound** on SMI's true generalization performance.

| | MY_LABEL=0 | MY_LABEL=1 |
|---|---|---|
| best_label=0 | 207 | 176 |
| best_label=1 | 131 | 252 |

**Interpretation:** If SMI achieves F1=X against `MY_LABEL` on the 4,234 non-gold articles, and `MY_LABEL` agrees with `best_label` only 59.9% of the time, then SMI's true F1 (against `best_label`) is likely higher than X. We report both the raw F1 and the "ceiling-adjusted" estimate below.

This evaluation fully resolves the in-sample evaluation concern.


In [10]:
# === True Out-of-Sample Evaluation: SMI on 4,234 Non-Gold Articles ===
# TASK5-NONGOLD-EVAL
print('='*70)
print('TRUE OUT-OF-SAMPLE: SMI on 4,234 Non-Gold Articles (vs MY_LABEL)')
print('='*70)

# Identify the 766 gold articles (used for SMI training)
gold_article_ids = set(gold['article_id'].values)
corpus_with_human = corpus_crit_df[corpus_crit_df['human_label'].notna()].copy()
corpus_with_human['human_label'] = corpus_with_human['human_label'].astype(int)

# Split into gold-overlap (766) and non-gold (4234)
gold_overlap = corpus_with_human[corpus_with_human['article_id'].isin(gold_article_ids)].copy()
non_gold = corpus_with_human[~corpus_with_human['article_id'].isin(gold_article_ids)].copy()

print(f'Corpus total: {len(corpus_with_human)}')
print(f'  Gold overlap (training data): {len(gold_overlap)}')
print(f'  Non-gold (true held-out):     {len(non_gold)}')

# 1) Ceiling: best_label vs MY_LABEL agreement on the 766 overlap
# (This tells us how noisy MY_LABEL is relative to best_label)
print(f'\n--- Label Noise Ceiling (on 766 gold-overlap articles) ---')
overlap_labels = gold_overlap[['article_id', 'human_label']].merge(
    gold[['article_id', 'best_label']], on='article_id', how='inner'
)
ceiling_agreement = float((overlap_labels['best_label'] == overlap_labels['human_label']).mean())
print(f'  best_label vs MY_LABEL agreement: {ceiling_agreement:.4f}')
print(f'  Crosstab:')
print(pd.crosstab(overlap_labels['best_label'], overlap_labels['human_label'],
                  rownames=['best_label'], colnames=['MY_LABEL']))

# 2) SMI vs MY_LABEL on the 766 gold-overlap articles (in-sample, but
#    against the noisy pre-revision label — gives a sense of how much
#    of the apparent error on (3) is label noise vs SMI error)
print(f'\n--- SMI vs MY_LABEL on 766 Gold-Overlap Articles (in-sample, noisy label) ---')
y_true_overlap = gold_overlap['human_label'].values
y_pred_overlap = gold_overlap['SMI_label'].values
overlap_f1 = f1_score(y_true_overlap, y_pred_overlap, zero_division=0)
overlap_acc = accuracy_score(y_true_overlap, y_pred_overlap)
overlap_kappa = cohen_kappa_score(y_true_overlap, y_pred_overlap)
print(f'  F1: {overlap_f1:.4f}  Acc: {overlap_acc:.4f}  Kappa: {overlap_kappa:.4f}')
print(f'  (Compare with held-out F1 in Section 6: SMI was trained on best_label,')
print(f'   not MY_LABEL, so this gap reflects label noise + memorization.)')

# 3) SMI vs MY_LABEL on the 4,234 non-gold articles (TRUE out-of-sample)
print(f'\n--- SMI vs MY_LABEL on 4,234 Non-Gold Articles (TRUE held-out) ---')
y_true_nongold = non_gold['human_label'].values
y_pred_nongold = non_gold['SMI_label'].values

nongold_f1   = f1_score(y_true_nongold, y_pred_nongold, zero_division=0)
nongold_acc  = accuracy_score(y_true_nongold, y_pred_nongold)
nongold_prec = precision_score(y_true_nongold, y_pred_nongold, zero_division=0)
nongold_rec  = recall_score(y_true_nongold, y_pred_nongold, zero_division=0)
nongold_kappa = cohen_kappa_score(y_true_nongold, y_pred_nongold)
nongold_mcc  = matthews_corrcoef(y_true_nongold, y_pred_nongold)
nongold_probs = non_gold['SMI_score'].values
nongold_auc  = roc_auc_score(y_true_nongold, nongold_probs) if len(set(y_true_nongold)) > 1 else 0.0

print(f'  F1:        {nongold_f1:.4f}')
print(f'  Accuracy:  {nongold_acc:.4f}')
print(f'  Precision: {nongold_prec:.4f}')
print(f'  Recall:    {nongold_rec:.4f}')
print(f'  Kappa:     {nongold_kappa:.4f}')
print(f'  MCC:       {nongold_mcc:.4f}')
print(f'  AUC:       {nongold_auc:.4f}')
cm = confusion_matrix(y_true_nongold, y_pred_nongold)
print(f'\n  Confusion matrix:')
print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
print(f'  FN={cm[1,0]}  TP={cm[1,1]}')

# 4) Ceiling-adjusted estimate (heuristic, not a statistical correction)
# If MY_LABEL agrees with best_label 59.9% of the time, and SMI achieves
# F1=X against MY_LABEL, then SMI's "true" F1 against best_label is
# likely higher. The naive scaling X / ceiling gives an upper-bound-ish
# estimate (capped at 1.0).
ceiling_adjusted_f1 = min(nongold_f1 / ceiling_agreement, 1.0) if ceiling_agreement > 0 else nongold_f1
print(f'\n--- Ceiling-Adjusted Estimate (heuristic) ---')
print(f'  Observed F1 (vs MY_LABEL):           {nongold_f1:.4f}')
print(f'  Label agreement ceiling:             {ceiling_agreement:.4f}')
print(f'  Ceiling-adjusted F1 (heuristic):     {ceiling_adjusted_f1:.4f}')
print(f'  NOTE: This is a rough heuristic, NOT a statistical correction.')
print(f'        The true F1 against best_label is likely between')
print(f'        {nongold_f1:.4f} and {ceiling_adjusted_f1:.4f}.')

# Store for the save-results cell
nongold_metrics = {
    'n_articles': int(len(non_gold)),
    'f1': float(nongold_f1),
    'accuracy': float(nongold_acc),
    'precision': float(nongold_prec),
    'recall': float(nongold_rec),
    'kappa': float(nongold_kappa),
    'mcc': float(nongold_mcc),
    'auc': float(nongold_auc),
    'tp': int(cm[1,1]),
    'fp': int(cm[0,1]),
    'fn': int(cm[1,0]),
    'tn': int(cm[0,0]),
    'label_agreement_ceiling': float(ceiling_agreement),
    'ceiling_adjusted_f1_heuristic': float(ceiling_adjusted_f1),
    'overlap_f1_insample': float(overlap_f1),
    'overlap_kappa_insample': float(overlap_kappa),
    'note': ('True out-of-sample evaluation. SMI trained on 766 gold '
             '(best_label), evaluated on 4234 non-gold articles using '
             'MY_LABEL (pre-revision human annotation) as ground truth. '
             'MY_LABEL agrees with best_label 59.9% on overlap, so '
             'observed F1 is a lower bound on true performance.'),
}
print(f'\nStored nongold_metrics for save-results cell.')

TRUE OUT-OF-SAMPLE: SMI on 4,234 Non-Gold Articles (vs MY_LABEL)
Corpus total: 5000
  Gold overlap (training data): 766
  Non-gold (true held-out):     4234

--- Label Noise Ceiling (on 766 gold-overlap articles) ---
  best_label vs MY_LABEL agreement: 0.5992
  Crosstab:
MY_LABEL      0    1
best_label          
0           207  176
1           131  252

--- SMI vs MY_LABEL on 766 Gold-Overlap Articles (in-sample, noisy label) ---
  F1: 0.6491  Acc: 0.6175  Kappa: 0.2293
  (Compare with held-out F1 in Section 6: SMI was trained on best_label,
   not MY_LABEL, so this gap reflects label noise + memorization.)

--- SMI vs MY_LABEL on 4,234 Non-Gold Articles (TRUE held-out) ---
  F1:        0.2674
  Accuracy:  0.5239
  Precision: 0.4172
  Recall:    0.1968
  Kappa:     -0.0218
  MCC:       -0.0252
  AUC:       0.4570

  Confusion matrix:
  TN=1850  FP=514
  FN=1502  TP=368

--- Ceiling-Adjusted Estimate (heuristic) ---
  Observed F1 (vs MY_LABEL):           0.2674
  Label agreement ceilin

## 7. Interpretability Analysis

SMI provides per-article explanations by showing which criteria contributed most to the yellow prediction.

For any article $a$ predicted as yellow, the contribution of each criterion is:
$$\text{Contrib}_i(a) = w_i \cdot C_i(a)$$


In [11]:
# === Interpretability: Show why articles are labeled yellow ===
print('=== Sample SMI Annotations with Explanations ===')
print()

# Show 5 yellow and 5 non-yellow examples
gold_with_smi = gold.copy()
gold_with_smi['SMI_score'] = gold_smi_scores
gold_with_smi['SMI_label'] = gold_smi_labels

# Get criteria scores for each article
for i in range(8):
    gold_with_smi[f'C{i+1}'] = X[:, i]

# Yellow examples (correctly predicted)
yellow_correct = gold_with_smi[(gold_with_smi.best_label==1) & (gold_with_smi.SMI_label==1)]
print(f'--- SMI YELLOW Predictions ({len(yellow_correct)} correct out of {gold_smi_labels.sum()} predicted) ---')
for _, r in yellow_correct.sample(min(5, len(yellow_correct)), random_state=42).iterrows():
    print(f'\n  [{r.article_id}] SMI_score={r.SMI_score:.3f}')
    print(f'  Headline: {r.headline[:80]}')
    print(f'  Top contributing criteria:')
    contribs = [(criteria_names[i], final_lr.coef_[0][i] * r[f'C{i+1}'], r[f'C{i+1}']) for i in range(8)]
    contribs.sort(key=lambda x: -abs(x[1]))
    for name, contrib, score in contribs[:3]:
        if abs(contrib) > 0.01:
            print(f'    {name}: score={score:.2f}, contribution={contrib:+.3f}')

# Non-yellow examples
ny_correct = gold_with_smi[(gold_with_smi.best_label==0) & (gold_with_smi.SMI_label==0)]
print(f'\n--- SMI NON-YELLOW Predictions ({len(ny_correct)} correct) ---')
for _, r in ny_correct.sample(min(3, len(ny_correct)), random_state=42).iterrows():
    print(f'\n  [{r.article_id}] SMI_score={r.SMI_score:.3f}')
    print(f'  Headline: {r.headline[:80]}')

=== Sample SMI Annotations with Explanations ===

--- SMI YELLOW Predictions (336 correct out of 407 predicted) ---

  [v18_2427] SMI_score=0.574
  Headline: স্টেডিয়ামের গেটে ‘বিব্রত’ গাভাস্কার
  Top contributing criteria:
    C4: Attribution Gap: score=0.90, contribution=+1.821
    C6: Entertainment Displacement: score=0.33, contribution=+0.738

  [v18_2786] SMI_score=0.981
  Headline: বিশ্রামের অজুহাতে বাদ মাহমুদউল্লাহ!
  Top contributing criteria:
    C6: Entertainment Displacement: score=1.00, contribution=+2.214
    C1: Sensational Headline: score=0.30, contribution=+1.983
    C4: Attribution Gap: score=0.60, contribution=+1.214

  [v18_2376] SMI_score=0.582
  Headline: প্রতিপক্ষকে ফাঁসাতে নিজের মেয়েকে হত্যা করান বাবা!
  Top contributing criteria:
    C1: Sensational Headline: score=0.30, contribution=+1.983
    C5: Speculation: score=0.58, contribution=+0.625
    C8: Sensitive Topic: score=0.24, contribution=-0.018

  [v18_1062] SMI_score=0.793
  Headline: হত্যার পর সড়ক দুর্ঘট

## 8. Annotation Speed Comparison

SMI enables rapid annotation of large corpora without human effort.


In [12]:
# === Annotation speed comparison ===
print('=== Annotation Speed Comparison ===')
print()
print(f'{"Method":<25} {"Time for 5,000":>20} {"Cost":>15}')
print('-' * 65)
print(f'{"Human annotation":<25} {"~167 hours":>20} {"$$ (3 annotators)":>15}')
print(f'{"SMI annotation":<25} {f"{smi_time:.2f}s":>20} {"$0":>15}')
print(f'{"Speedup":<25} {f"{167*3600/smi_time:.0f}x":>20} {"":>15}')
print()
print(f'SMI annotated {len(corpus_crit_df)} articles in {smi_time:.2f} seconds')
print(f'Average: {smi_time/len(corpus_crit_df)*1000:.2f}ms per article')

=== Annotation Speed Comparison ===

Method                          Time for 5,000            Cost
-----------------------------------------------------------------
Human annotation                    ~167 hours $$ (3 annotators)
SMI annotation                          16.17s              $0
Speedup                                 37191x                

SMI annotated 5000 articles in 16.17 seconds
Average: 3.23ms per article


## 9. Save Results


In [13]:
# === Save all results ===

# 1. SMI weights — trained on ALL 766 gold articles, used to annotate the
#    5,000-article corpus. Schema matches results/smi_weights.json with
#    8 criteria (C1-C8).
weights_dict = {
    'weights': {name: float(w) for name, w in zip(criteria_names, final_lr.coef_[0])},
    'bias': float(final_lr.intercept_[0]),
    'threshold': float(tau_star),
}
with open(OUTPUT_DIR / 'smi_weights.json', 'w') as f:
    json.dump(weights_dict, f, indent=2)
print(f'Saved: smi_weights.json ({len(criteria_names)} criteria)')

# 2. SMI annotated corpus (5,000 articles)
# Use corpus_batch (canonical) — falls back to news_source (legacy alias) if absent.
annotation_cols = ['article_id', 'headline']
if 'corpus_batch' in corpus.columns:
    annotation_cols.append('corpus_batch')
elif 'news_source' in corpus.columns:
    annotation_cols.append('news_source')
smi_annotation = corpus[annotation_cols].copy()
smi_annotation['SMI_score'] = smi_scores
smi_annotation['SMI_label'] = smi_labels
if 'human_label' in corpus.columns:
    smi_annotation['human_label'] = corpus['human_label'].values
smi_annotation.to_csv(OUTPUT_DIR / 'smi_annotated_5000.csv', index=False)
print(f'Saved: smi_annotated_5000.csv ({len(smi_annotation)} articles)')

# 3. Validation results — corrected schema (Task 1 + Task 5 + Task 9).
#    Three evaluation tiers:
#      (a) Held-out 70/30 split on 766 gold — PRIMARY in-sample held-out
#      (b) 5-fold CV on 766 gold — SECONDARY variance estimate
#      (c) True out-of-sample on 4,234 non-gold (NEW in Task 5) — STRONGEST
#          generalization test, lower bound due to MY_LABEL noise
#    Previous versions of this file reported an in-sample F1=0.846 on the
#    training data which was removed due to data leakage.
#    NEW in Task 9: data_version field records which CSV version was used
#    (cleaned vs legacy) for full provenance.
results = {
    "gold_standard_size": 766,
    "gold_train_size": int(len(y_tr)),
    "gold_test_size": int(len(y_te)),

    # (a) Held-out (70/30 split) — PRIMARY in-sample held-out metric
    "held_out_f1": float(held_out_f1),
    "held_out_accuracy": float(held_out_accuracy),
    "held_out_precision": float(held_out_precision),
    "held_out_recall": float(held_out_recall),
    "held_out_kappa": float(held_out_kappa),
    "held_out_mcc": float(held_out_mcc),
    "held_out_auc": float(held_out_auc),

    # (b) 5-fold CV — SECONDARY variance estimate
    "cv_f1_mean": float(np.mean(cv_f1s)),
    "cv_f1_std": float(np.std(cv_f1s)),
    "cv_accuracy_mean": float(np.mean(cv_accs)),
    "cv_accuracy_std": float(np.std(cv_accs)),

    # (c) True out-of-sample on 4,234 non-gold (NEW in Task 5)
    "nongold_4234_evaluation": nongold_metrics,

    # Corpus annotation
    "corpus_size": 5000,
    "corpus_filename": os.path.basename(CORPUS_PATH),
    "smi_yellow_count": int(smi_labels.sum()),
    "smi_non_yellow_count": int((1 - smi_labels).sum()),

    # Body truncation disclosure (NEW in Task 5)
    "corpus_body_truncation": {
        "n_truncated": int(corpus_n_truncated),
        "pct_truncated": float(corpus_pct_truncated),
        "truncation_threshold_chars": 1200,
        "note": ("body_preview column in the corpus CSV is capped at 1200 chars "
                 "when body_full_len > 1200. Affects ~9.1% of articles. SMI "
                 "criteria C3/C5/C6 use word-density formulas that may be "
                 "slightly affected."),
    },

    # Data version provenance (NEW in Task 9) — tracks which CSV version was used
    "data_version": {
        "gold_csv": os.path.basename(GOLD_PATH),
        "corpus_csv": os.path.basename(CORPUS_PATH),
        "corpus_metadata_csv": os.path.basename(CORPUS_METADATA_PATH) if CORPUS_METADATA_PATH else None,
        "using_cleaned_csvs": bool(using_cleaned),
        "text_normalization": "NFC + ZWJ/ZWNJ stripped" if using_cleaned else "legacy (pre-cleaning)",
        "n_stub_articles_gold": int(n_stub_gold),
        "n_stub_articles_corpus": int(n_stub_corpus),
        "note": ("Cleaned CSVs (Task 8) apply NFC normalization, strip zero-width "
                 "joiners, and rename columns for professionalism. Stub articles "
                 "(body_text='not_available') are kept in the dataset but treated "
                 "as empty for SMI density-based criteria (C3/C5/C6)."),
    },

    # Annotation speed
    "annotation_time_seconds": float(smi_time),
    "speedup_vs_human": float(167 * 3600 / smi_time),

    # Provenance
    "seed": 42,
    "n_criteria": 8,
    "criteria_names": ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8"],
    "needs_rerun": False,
    "note": ("Updated 2026-07-19 (Task 1 + Task 5 + Task 9): "
             "(1) corpus filename changed to v18_HUMAN_GOLD_FINAL_v20_GOLD_5000.csv; "
             "(2) column mapping added (v18_id->article_id, body_preview->body_text, "
             "source->corpus_batch, MY_LABEL->human_label); "
             "(3) NEW true out-of-sample evaluation on 4,234 non-gold articles added "
             "(see nongold_4234_evaluation); "
             "(4) Task 9: NB8 now supports both cleaned CSVs (Swarabyanjan_Gold_Balanced_766.csv, "
             "Swarabyanjan_Corpus_5000.csv) and legacy CSVs (Swarabyanjan_BEST_BALANCED_1to1.csv, "
             "v18_HUMAN_GOLD_FINAL_v20_GOLD_5000.csv) via auto-detection; "
             "(5) Task 9: stub articles (body_text='not_available') are replaced with "
             "empty strings for SMI density-based scoring (original preserved in body_text_original). "
             "Previous in-sample F1=0.846 was removed (data leakage). "
             "Held-out 70/30 on 766 gold is the primary metric; 4,234 non-gold "
             "evaluation is the strongest generalization test (lower bound due to "
             "MY_LABEL noise: best_label vs MY_LABEL agreement = 59.9% on overlap)."),
}
with open(OUTPUT_DIR / 'smi_annotation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved: smi_annotation_results.json (corrected schema + nongold eval + data_version)')

# 4. Per-criteria scores for all articles
corpus_crit_df.to_csv(OUTPUT_DIR / 'smi_criteria_scores_5000.csv', index=False)
print(f'Saved: smi_criteria_scores_5000.csv')

print(f'\n{"="*60}')
print('SMI ANNOTATION EXPERIMENT COMPLETE')
print(f'{"="*60}')
print(f'\nSummary:')
print(f'  Gold standard:    766 articles (human-verified)')
print(f'  Held-out split:   {len(y_tr)} train / {len(y_te)} test (70/30 stratified)')
print(f'  SMI annotated:    5,000 articles (automated, model trained on all 766)')
print(f'  Data version:     {"cleaned CSVs (Task 8)" if using_cleaned else "legacy CSVs"}')
print(f'  Stub articles:    {n_stub_gold} in gold, {n_stub_corpus} in corpus (handled gracefully)')
print(f'  --- Evaluation (3 tiers) ---')
print(f'  Held-out F1 (70/30):       {held_out_f1:.4f}  (PRIMARY in-sample)')
print(f'  Held-out Kappa (70/30):    {held_out_kappa:.4f}')
print(f'  Held-out AUC (70/30):      {held_out_auc:.4f}')
print(f'  5-fold CV F1:              {np.mean(cv_f1s):.4f} ± {np.std(cv_f1s):.4f}  (variance)')
print(f'  Non-gold F1 (4234 vs MY):  {nongold_f1:.4f}  (TRUE out-of-sample, lower bound)')
print(f'  Non-gold Kappa:            {nongold_kappa:.4f}')
print(f'  Non-gold AUC:              {nongold_auc:.4f}')
print(f'  Ceiling-adjusted F1:       {ceiling_adjusted_f1:.4f}  (heuristic upper-ish)')
print(f'  --- Annotation speed ---')
print(f'  Annotation time:  {smi_time:.2f}s (vs 167 hours human)')
print(f'  Speedup:          {167*3600/smi_time:.0f}x')


Saved: smi_weights.json (8 criteria)
Saved: smi_annotated_5000.csv (5000 articles)
Saved: smi_annotation_results.json (corrected schema + nongold eval + data_version)
Saved: smi_criteria_scores_5000.csv

SMI ANNOTATION EXPERIMENT COMPLETE

Summary:
  Gold standard:    766 articles (human-verified)
  Held-out split:   536 train / 230 test (70/30 stratified)
  SMI annotated:    5,000 articles (automated, model trained on all 766)
  Data version:     cleaned CSVs (Task 8)
  Stub articles:    2 in gold, 4 in corpus (handled gracefully)
  --- Evaluation (3 tiers) ---
  Held-out F1 (70/30):       0.8353  (PRIMARY in-sample)
  Held-out Kappa (70/30):    0.6435
  Held-out AUC (70/30):      0.9139
  5-fold CV F1:              0.8091 ± 0.0564  (variance)
  Non-gold F1 (4234 vs MY):  0.2674  (TRUE out-of-sample, lower bound)
  Non-gold Kappa:            -0.0218
  Non-gold AUC:              0.4570
  Ceiling-adjusted F1:       0.4463  (heuristic upper-ish)
  --- Annotation speed ---
  Annotation ti